In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, get_dataset_config_names
from typing import Generator, Dict, Any

# ============================================================================
# ✨ 프로젝트 제목: 나만의 음성 인식 능력 측정기 (ASR 데이터 탐험)
# 🎧 데이터셋: hf-audio/open-asr-leaderboard
# 💡 데이터셋 설명: 이 데이터셋은 다양한 음성 인식(Automatic Speech Recognition, ASR)
#   경진대회의 결과물과 데이터를 모아둔 방대하고 재미있는 음성 데이터셋입니다.
#   쉽게 말해, "녹음된 오디오 파일"과 "그 오디오에 맞는 실제 대본(텍스트)"이 쌍으로 존재하는 데이터입니다.
# 🏆 우리가 목표할 것: 오디오가 얼마나 긴지(시간)와 텍스트가 얼마나 많은지(단어 수) 사이의 재미있는 관계를 분석해봅시다!
# ============================================================================

# 사용할 데이터셋 ID와 스플릿 지정
DATASET_NAME = "hf-audio/open-asr-leaderboard"
SPLIT_NAME = "test" # 테스트 셋을 사용해 전반적인 성능을 점검해 봅시다!
SAMPLE_COUNT = 15  # 친절하게, 데이터 15개만 가지고 재미있는 실습을 진행해 볼게요.

# 1. 설정 확인 및 데이터 로드
print("=" * 70)
print(f"🤖 [1/5] 데이터셋 '{DATASET_NAME}'을 로드할 준비를 합니다.")
print("=" * 70)

try:
    # 데이터셋의 사용 가능한 설정(Config) 이름 확인
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
except Exception as e:
    print(f"ℹ️ Config 로드 중 오류 발생: {e}. 기본 설정으로 진행합니다.")
    configs = []

# 스트리밍 모드 시도 (데이터셋 전체를 지연 다운로드하여 빠르게 탐색)
dataset = None
try:
    dataset = load_dataset(DATASET_NAME, name=configs[0] if configs else None, split=SPLIT_NAME, streaming=True)
    print("🚀 성공! 스트리밍(streaming=True) 모드로 데이터셋을 로드했습니다. (메모리 절약 짱!)")
    streaming_success = True
except Exception as e:
    print(f"\n⚠️ 경고: 스트리밍 로드 실패 ({e}). 대신 소량 다운로드 모드로 전환합니다.")
    # 스트리밍 실패 시, 소량 다운로드가 가능한 형태로 fallback
    try:
        dataset = load_dataset(DATASET_NAME, name=configs[0] if configs else None, split=SPLIT_NAME, streaming=False)
        print("💾 성공! 소량 다운로드 모드(streaming=False)로 데이터셋을 로드했습니다.")
        streaming_success = False
    except Exception as e_fallback:
        print(f"❌ 치명적 오류: 데이터셋 로드에 실패했습니다. 오류: {e_fallback}")
        exit()


# 2. 데이터 샘플링 준비 (반드시 이 패턴을 사용해야 합니다!)
print("\n" + "=" * 70)
print(f"✨ [2/5] {SAMPLE_COUNT}개의 샘플만 골라서 탐험을 시작할 거예요!")

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)인 경우
    print("🔗 스트리밍 모드 감지: .take() 사용.")
    sample_iterator = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)인 경우
    print("📦 일반 모드 감지: Dataset.take() 사용.")
    sample_iterator = iter(dataset.take(SAMPLE_COUNT))

# 3. 데이터 분석 루프 (단어 수 vs. 시간 분석)
print("\n" + "=" * 70)
print("🔍 [3/5] 데이터 탐색: 오디오 길이와 텍스트 단어 수를 세어봅시다!")

# 결과를 임시 리스트에 저장할 공간을 만듭니다.
word_counts = []
audio_lengths = []
all_texts = []

sample_data_list = [] # 데이터를 하나씩 받아서 처리하기 위해 임시로 리스트에 저장

try:
    # 이터레이터(sample_iterator)를 통해 샘플을 하나씩 가져옵니다.
    print("🧠 AI가 열심히 데이터를 분석하고 있어요...")
    for i in range(SAMPLE_COUNT):
        try:
            # next() 함수를 사용하여 다음 샘플을 꺼내옵니다.
            sample = next(sample_iterator)
        except StopIteration:
            # 샘플이 다 떨어지면 루프 종료
            break
        
        # 💡 핵심 데이터 추출: 텍스트와 오디오 길이 가져오기
        text_content = sample['text']
        audio_len = sample['audio_length_s']
        
        # 📝 텍스트 전처리 및 단어 수 계산 (AI의 가장 기본 능력!)
        # 공백을 기준으로 쪼갠 후 개수를 셉니다.
        word_count = len(text_content.split())
        
        # 📊 결과를 리스트에 저장합니다.
        word_counts.append(word_count)
        audio_lengths.append(audio_len)
        all_texts.append(text_content)
        
        sample_data_list.append({
            'id': sample['id'],
            'audio_length_s': audio_len,
            'text': text_content,
            'word_count': word_count
        })
        
        print(f"  [샘플 {i+1}/{min(SAMPLE_COUNT, i+1)}] ID: {sample['id']} | 시간: {audio_len:.2f}s | 단어 수: {word_count}개")

except Exception as e:
    print(f"🛑 데이터 처리 중 예기치 않은 오류가 발생했습니다: {e}")

# 4. 분석 결과 요약 및 시각화 (친절한 해석 첨부)
print("\n" + "=" * 70)
print("📊 [4/5] 분석 결과 요약 및 비주얼라이징!")

if not sample_data_list:
    print("😔 데이터를 하나도 가져오지 못했어요. 로드된 데이터셋을 확인해주세요.")
else:
    # 📝 분석 결과 요약
    avg_time = np.mean(audio_lengths)
    avg_words = np.mean(word_counts)
    print(f"✅ 분석된 {len(sample_data_list)}개 샘플의 평균 정보:")
    print(f"   - 평균 오디오 길이: {avg_time:.2f} 초")
    print(f"   - 평균 단어 수: {avg_words:.2f} 개")
    
    print("\n💡 AI 튜터의 Insight:")
    print("   - 일반적으로 오디오 시간이 길수록 단어 수도 비례해서 많아지죠. 이 데이터셋도 그런 추세를 따르는지 한번 그래프로 확인해 봅시다!")
    
    # 📈 시각화 (matplotlib 사용)
    plt.figure(figsize=(10, 6))
    plt.scatter(audio_lengths, word_counts, alpha=0.6, s=100)
    
    # 추세선 추가 (데이터의 관계를 한눈에 보여주기)
    plt.plot(audio_lengths, word_counts, color='red', linestyle='--', alpha=0.5, label='Trend Line')
    
    plt.title('Audio Length vs. Word Count in ASR Data', fontsize=14)
    plt.xlabel('Audio Length (Seconds)', fontsize=12)
    plt.ylabel('Word Count (Words)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

# 5. 마무리
print("\n" + "=" * 70)
print("🎉 [5/5] 실습 완료! 정말 잘하셨어요!")
print("축하합니다! 당신은 데이터셋의 구조를 파악하고, 핵심 메타데이터(시간, 텍스트)를 추출하여 관계를 분석하는 능력을 배웠습니다.")
print("이 과정을 통해 실제로 AI 모델을 학습시키기 전에 데이터의 품질을 점검하는 중요한 '탐색적 데이터 분석(EDA)' 능력을 갖추게 되었답니다. 👍")
print("======================================================================")